<a href="https://colab.research.google.com/github/rahilkhan-acadmic/APAIML-GradedMiniProject/blob/develop/capstone/Phase5_API_Integration.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [22]:
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field
import joblib
import json
import pandas as pd
import datetime
import hashlib
import uvicorn
import nest_asyncio
import threading

app = FastAPI(title="Claim Outcome Prediction API", version="1.0.0")

#Load artifacts from google drive
import io
from google.colab import drive
drive.mount('/content/drive')
random_forest_model = joblib.load('/content/drive/MyDrive/files/capstone/random_forest_model.joblib')
feature_schema = json.load(open('/content/drive/MyDrive/files/capstone/feature_schema.json', 'r'))

# Request Schema Validation
class ClaimRequest(BaseModel):
    patient_id: int | None = Field(None, description="Optional patient identifier for tracking purposes") # Added patient_id
    age: int
    chronic_flag: int
    billed_amount: float
    approved_amount: float
    payment_days: float
    amount_difference: float
    approval_ratio: float
    days_between_visit_and_billing: int
    length_of_stay_days: float
    gender_M: bool
    city_Chennai: bool
    city_Delhi: bool
    city_Hyderabad: bool
    city_Mumbai: bool
    city_Pune: bool
    insurance_provider_HealthPlus: bool
    insurance_provider_MediCareX: bool
    insurance_provider_SecureLife: bool
    department_ER: bool
    department_General: bool
    department_ICU: bool
    department_Neurology: bool
    department_Orthopedics: bool
    visit_type_ICU: bool
    visit_type_OPD: bool
    avg_days_between_visits: float

@app.get("/health")
def health_check():
    return {"status": "healthy", "timestamp": datetime.datetime.now().isoformat()}

@app.post("/predict")
def predict(request: ClaimRequest):
    try:
        # Convert request to DataFrame
        # Exclude patient_id from the DataFrame used for prediction as it's for tracking only
        input_data_dict = request.model_dump(exclude={'patient_id'})
        input_data = pd.DataFrame([input_data_dict])

        # Log input feature hash for auditability
        feature_hash = hashlib.sha256(str(input_data_dict).encode()).hexdigest()

        # Ensure column order matches feature schema
        input_data = input_data[feature_schema]

        # Generate prediction
        prediction_idx = int(random_forest_model.predict(input_data)[0])
        mapping = {0: "Paid", 1: "Pending", 2: "Rejected"}
        prediction_label = mapping.get(prediction_idx, "Unknown")

        # Log Prediction
        log_entry = {}
        if request.patient_id is not None:
            log_entry["patient_id"] = request.patient_id # Include patient_id in response first

        log_entry.update({
            "timestamp": datetime.datetime.now().isoformat(),
            "model_version": "1.0.0",
            "input_hash": feature_hash,
            "prediction": prediction_label
        })

        return log_entry
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

# Apply nest_asyncio to allow uvicorn to run in a Colab environment
nest_asyncio.apply()

def run_uvicorn():
    uvicorn.run(app, host="0.0.0.0", port=8000)

# Start uvicorn in a separate thread to not block the Colab kernel
uvicorn_thread = threading.Thread(target=run_uvicorn)
uvicorn_thread.daemon = True # Allow the program to exit even if the thread is still running
uvicorn_thread.start()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


INFO:     Started server process [6124]
INFO:     Waiting for application startup.
INFO:     Application startup complete.


In [23]:
#code to check health using /health
import requests

# The API endpoint
url = "http://localhost:8000/health"

# A GET request to the API
response = requests.get(url)

# Print the response
print(response.json())

ERROR:    [Errno 98] error while attempting to bind on address ('0.0.0.0', 8000): address already in use
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.


INFO:     127.0.0.1:36864 - "GET /health HTTP/1.1" 200 OK
{'status': 'healthy', 'timestamp': '2026-03-07T06:09:05.224507'}


In [24]:
# A list of sample payloads for batch prediction
payloads = [
    {
        "patient_id": 1,
        "age": 45,
        "chronic_flag": 0,
        "billed_amount": 1500.75,
        "approved_amount": 1200.50,
        "payment_days": 30.0,
        "amount_difference": 300.25,
        "approval_ratio": 0.8,
        "days_between_visit_and_billing": 10,
        "length_of_stay_days": 3.5,
        "gender_M": True,
        "city_Chennai": False,
        "city_Delhi": True,
        "city_Hyderabad": False,
        "city_Mumbai": False,
        "city_Pune": False,
        "insurance_provider_HealthPlus": True,
        "insurance_provider_MediCareX": False,
        "insurance_provider_SecureLife": False,
        "department_ER": False,
        "department_General": True,
        "department_ICU": False,
        "department_Neurology": False,
        "department_Orthopedics": False,
        "visit_type_ICU": False,
        "visit_type_OPD": True,
        "avg_days_between_visits": 45.0
    },
    {
        "patient_id": 2,
        "age": 60,
        "chronic_flag": 1,
        "billed_amount": 3000.00,
        "approved_amount": 2800.00,
        "payment_days": 60.0,
        "amount_difference": 200.00,
        "approval_ratio": 0.93,
        "days_between_visit_and_billing": 20,
        "length_of_stay_days": 7.0,
        "gender_M": False,
        "city_Chennai": True,
        "city_Delhi": False,
        "city_Hyderabad": False,
        "city_Mumbai": False,
        "city_Pune": False,
        "insurance_provider_HealthPlus": False,
        "insurance_provider_MediCareX": True,
        "insurance_provider_SecureLife": False,
        "department_ER": True,
        "department_General": False,
        "department_ICU": False,
        "department_Neurology": False,
        "department_Orthopedics": False,
        "visit_type_ICU": False,
        "visit_type_OPD": False,
        "avg_days_between_visits": 90.0
    },
    {
        "patient_id": 3,
        "age": 28,
        "chronic_flag": 0,
        "billed_amount": 500.00,
        "approved_amount": 500.00,
        "payment_days": 15.0,
        "amount_difference": 0.00,
        "approval_ratio": 1.0,
        "days_between_visit_and_billing": 5,
        "length_of_stay_days": 1.0,
        "gender_M": True,
        "city_Chennai": False,
        "city_Delhi": False,
        "city_Hyderabad": False,
        "city_Mumbai": True,
        "city_Pune": False,
        "insurance_provider_HealthPlus": False,
        "insurance_provider_MediCareX": False,
        "insurance_provider_SecureLife": True,
        "department_ER": False,
        "department_General": True,
        "department_ICU": False,
        "department_Neurology": False,
        "department_Orthopedics": False,
        "visit_type_ICU": False,
        "visit_type_OPD": True,
        "avg_days_between_visits": 30.0
    }
]

Now that you have a list of payloads, you can iterate through it to make predictions for each one. Here's how you can do it:

In [25]:
import requests

url = "http://localhost:8000/predict"
predictions = []

for payload in payloads:
    try:
        prediction_response = requests.post(url, json=payload)
        prediction_response.raise_for_status()  # Raise an exception for HTTP errors

        # Get the JSON content and add patient_id to it
        prediction_data = prediction_response.json()
        # if 'patient_id' not in prediction_data and 'patient_id' in payload:
        #     pass

        predictions.append(prediction_data)
    except requests.exceptions.RequestException as e:
        print(f"Error making prediction: {e}")
        predictions.append({"error": str(e), "payload": payload})

display(predictions)

INFO:     127.0.0.1:36866 - "POST /predict HTTP/1.1" 200 OK
INFO:     127.0.0.1:36882 - "POST /predict HTTP/1.1" 200 OK
INFO:     127.0.0.1:36890 - "POST /predict HTTP/1.1" 200 OK


[{'patient_id': 1,
  'timestamp': '2026-03-07T06:09:05.269359',
  'model_version': '1.0.0',
  'input_hash': '21d94b2abb5845f70404e02921d77c6942756480d0b4e75b0b432a6108b64438',
  'prediction': 'Pending'},
 {'patient_id': 2,
  'timestamp': '2026-03-07T06:09:05.284706',
  'model_version': '1.0.0',
  'input_hash': '2eaede6f940aa4d32a8696daf06ae0face1a644062afe7b15cc0ed45745ba449',
  'prediction': 'Pending'},
 {'patient_id': 3,
  'timestamp': '2026-03-07T06:09:05.303217',
  'model_version': '1.0.0',
  'input_hash': 'ca5a2bad7a6095314997d281e39701d8a3c23b95e86d48a9a5e08ce3fd2501a0',
  'prediction': 'Paid'}]